# Data Loading

In [215]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
from sklearn.svm import SVC


warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"


CPU times: total: 15.6 ms
Wall time: 4 ms


In [216]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3


# Training with SVC


## RBF Kernel

### Tie Breaking and Data Splitting

#### **Balanced train-test split based on Y**

In [217]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(
    cat_df, test_size=0.2,
    tie_tolerance=0.5  # algorithms within tie_tolerance% of min treated as ties
    )

# Label‐encode the string targets into integers 0…4  ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [233]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Instantiate & train SVC with RBF kernel
svc_strat_y = SVC(kernel='rbf', C=0.5, gamma='auto', probability=True, random_state=42)

svc_strat_y.fit(X_train_scaled, y_train, sample_weight=weights_train)



SVC(C=0.5, gamma='auto', probability=True, random_state=42)

Classification report and Feature Importances

In [234]:

# Reversing mapping from integer labels to algorithm names
y_pred_int = svc_strat_y.predict(X_test_scaled)
y_pred_str = le.inverse_transform(y_pred_int)


# Evaluate
y_pred = svc_strat_y.predict(X_test)
print("Classification Report:")
print(classification_report(y_test_str, y_pred_str, sample_weight=weights_test, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

       AltRB       0.07      0.08      0.08 1548.998634966601
    Pivot_Br       0.23      0.14      0.17 3764.789222512713
        S_Br       0.41      0.46      0.43 3152.339561854101
        UMKP       0.33      0.08      0.12 1449.3346678647963
        kPEX       0.25      0.40      0.30 3378.3977986771524

    accuracy                           0.27 13293.859885875361
   macro avg       0.26      0.23      0.22 13293.859885875361
weighted avg       0.27      0.27      0.25 13293.859885875361



#### Inference on 20% dataset

In [235]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=svc_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_RBF_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_RBF_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 46.67%

Precision by Algorithm:
  UMKP: 33.33% (24 predictions)
  AltRB: 28.57% (70 predictions)
  kPEX: 66.04% (106 predictions)
  Pivot_Br: 39.66% (58 predictions)
  S_Br: 46.30% (162 predictions)

Recall by Algorithm:
  UMKP: 44.90% (22/49 cases)
  AltRB: 50.00% (30/60 cases)
  kPEX: 52.56% (113/215 cases)
  Pivot_Br: 51.61% (80/155 cases)
  S_Br: 63.76% (95/149 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 64481.90
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.336
  Avg Overhead/case ((pred - opt) / opt)    = 321.93%


Adding prediction probabilities...

Saving full predictions to './SVC_RBF_results/tie_tol/50/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 224 wrong predictions with time metrics

Top 5 

#### Re-evalute on entire dataset

In [236]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=svc_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_RBF_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_RBF_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 47.89%

Precision by Algorithm:
  UMKP: 45.95% (37 predictions)
  AltRB: 40.62% (128 predictions)
  kPEX: 61.26% (191 predictions)
  Pivot_Br: 29.13% (127 predictions)
  S_Br: 50.90% (277 predictions)

Recall by Algorithm:
  UMKP: 45.00% (45/100 cases)
  AltRB: 61.21% (71/116 cases)
  kPEX: 50.69% (184/363 cases)
  Pivot_Br: 54.42% (123/226 cases)
  S_Br: 67.19% (172/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 106551.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.593
  Avg Overhead/case ((pred - opt) / opt)    = 491.02%


Adding prediction probabilities...

Saving full predictions to './SVC_RBF_results/tie_tol/50/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 396 wrong predictions with time metri

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

#### **Balanced train-test split based on both Y and k**

In [237]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2,
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )

# Label‐encode the string targets into integers 0…4  ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [248]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Instantiate & train SVC with RBF kernel
svc_strat_y_K = SVC(kernel='rbf', C=0.5, gamma='auto', probability=True, random_state=42)

svc_strat_y_K.fit(X_train_scaled, y_train, sample_weight=weights_train)



SVC(C=0.5, gamma='auto', probability=True, random_state=42)

Classification report and Feature Importances

In [249]:

# Reversing mapping from integer labels to algorithm names
y_pred_int = svc_strat_y_K.predict(X_test_scaled)
y_pred_str = le.inverse_transform(y_pred_int)


# Evaluate
y_pred = svc_strat_y_K.predict(X_test)
print("Classification Report:")
print(classification_report(y_test_str, y_pred_str, sample_weight=weights_test, zero_division=0))



Classification Report:
              precision    recall  f1-score   support

       AltRB       0.43      0.16      0.23 1483.3530488672318
    Pivot_Br       0.28      0.43      0.34 3582.073469039411
        S_Br       0.39      0.55      0.45 1849.663881911921
        UMKP       0.15      0.21      0.17 786.1821768006591
        kPEX       0.41      0.23      0.29 4679.182576808955

    accuracy                           0.33 12380.455153428178
   macro avg       0.33      0.32      0.30 12380.455153428178
weighted avg       0.35      0.33      0.32 12380.455153428178



#### Inference on 20% dataset

In [250]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=svc_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_RBF_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_RBF_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 44.05%

Precision by Algorithm:
  UMKP: 32.05% (78 predictions)
  AltRB: 53.33% (15 predictions)
  kPEX: 64.36% (101 predictions)
  Pivot_Br: 15.62% (64 predictions)
  S_Br: 47.53% (162 predictions)

Recall by Algorithm:
  UMKP: 77.36% (41/53 cases)
  AltRB: 42.03% (29/69 cases)
  kPEX: 46.64% (104/223 cases)
  Pivot_Br: 55.48% (86/155 cases)
  S_Br: 71.23% (104/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 71583.80
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.543
  Avg Overhead/case ((pred - opt) / opt)    = 264.55%


Adding prediction probabilities...

Saving full predictions to './SVC_RBF_results/tie_tol/50/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 235 wrong predictions with time metrics

Top

#### Re-evalute on entire dataset

In [251]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=svc_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_RBF_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_RBF_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 45.26%

Precision by Algorithm:
  UMKP: 31.30% (115 predictions)
  AltRB: 50.00% (50 predictions)
  kPEX: 59.11% (203 predictions)
  Pivot_Br: 17.76% (107 predictions)
  S_Br: 50.53% (285 predictions)

Recall by Algorithm:
  UMKP: 58.00% (58/100 cases)
  AltRB: 42.24% (49/116 cases)
  kPEX: 47.38% (172/363 cases)
  Pivot_Br: 51.77% (117/226 cases)
  S_Br: 71.88% (184/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 108785.20
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.627
  Avg Overhead/case ((pred - opt) / opt)    = 448.15%


Adding prediction probabilities...

Saving full predictions to './SVC_RBF_results/tie_tol/50/expand_and_split/all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 416 wrong predictions with time met

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


#### **Balanced train-test split based on Y**

In [252]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [253]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties

)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [257]:
# Instantiate & train SVC with RBF kernel
svc_holdout_strat_y = SVC(kernel='rbf', C=0.5, gamma='auto', probability=True, random_state=42)

svc_holdout_strat_y.fit(X_train_scaled, y_train_, sample_weight=weights_train_)


SVC(C=0.5, gamma='auto', probability=True, random_state=42)

#### Inference on 20% dataset

In [258]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=svc_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_RBF_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_RBF_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.64%

Precision by Algorithm:
  UMKP: 37.50% (8 predictions)
  AltRB: 17.24% (29 predictions)
  kPEX: 69.09% (55 predictions)
  Pivot_Br: 50.00% (2 predictions)
  S_Br: 51.61% (62 predictions)

Recall by Algorithm:
  UMKP: 35.00% (7/20 cases)
  AltRB: 43.48% (10/23 cases)
  kPEX: 61.64% (45/73 cases)
  Pivot_Br: 44.44% (20/45 cases)
  S_Br: 74.51% (38/51 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 21317.50
  Total Optimal Time   = 16592.90
  Time Ratio (pred/opt)= 1.285
  Avg Overhead/case ((pred - opt) / opt)    = 7431.22%


Adding prediction probabilities...

Saving full predictions to './SVC_RBF_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 77 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
           dataset  k_value      weigh

#### Re-evalute on entire dataset

In [259]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=svc_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_RBF_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_RBF_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 53.29%

Precision by Algorithm:
  UMKP: 42.00% (50 predictions)
  AltRB: 39.66% (116 predictions)
  kPEX: 65.41% (292 predictions)
  Pivot_Br: 23.08% (13 predictions)
  S_Br: 49.83% (289 predictions)

Recall by Algorithm:
  UMKP: 47.00% (47/100 cases)
  AltRB: 56.90% (66/116 cases)
  kPEX: 65.29% (237/363 cases)
  Pivot_Br: 47.79% (108/226 cases)
  S_Br: 70.31% (180/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 108939.40
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.629
  Avg Overhead/case ((pred - opt) / opt)    = 444.27%


Adding prediction probabilities...

Saving full predictions to './SVC_RBF_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 355 wrong predictions with time metrics



#### **Balanced train-test split based on Y and K**

In [260]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [276]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties

)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [277]:
# Instantiate & train SVC with RBF kernel
svc_holdout_strat_y_K = SVC(kernel='rbf', C=0.3, gamma='auto', probability=True, random_state=42)

svc_holdout_strat_y_K.fit(X_train_scaled, y_train_, sample_weight=weights_train_)


SVC(C=0.3, gamma='auto', probability=True, random_state=42)

#### Inference on 20% dataset

In [278]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=svc_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_RBF_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_RBF_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.70%

Precision by Algorithm:
  UMKP: 50.00% (12 predictions)
  AltRB: 30.00% (20 predictions)
  kPEX: 70.18% (57 predictions)
  Pivot_Br: 33.33% (6 predictions)
  S_Br: 45.28% (53 predictions)

Recall by Algorithm:
  UMKP: 70.00% (14/20 cases)
  AltRB: 52.17% (12/23 cases)
  kPEX: 65.75% (48/73 cases)
  Pivot_Br: 48.89% (22/45 cases)
  S_Br: 60.78% (31/51 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 23370.30
  Total Optimal Time   = 11002.60
  Time Ratio (pred/opt)= 2.124
  Avg Overhead/case ((pred - opt) / opt)    = 8359.35%


Adding prediction probabilities...

Saving full predictions to './SVC_RBF_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 70 wrong predictions with time metrics

Top 5 heaviest

#### Re-evalute on entire dataset

In [279]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=svc_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_RBF_results/tie_tol/50/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_RBF_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.24%

Precision by Algorithm:
  UMKP: 40.68% (59 predictions)
  AltRB: 40.78% (103 predictions)
  kPEX: 65.68% (271 predictions)
  Pivot_Br: 21.43% (42 predictions)
  S_Br: 50.53% (285 predictions)

Recall by Algorithm:
  UMKP: 51.00% (51/100 cases)
  AltRB: 54.31% (63/116 cases)
  kPEX: 62.26% (226/363 cases)
  Pivot_Br: 50.44% (114/226 cases)
  S_Br: 72.27% (185/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 110096.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.646
  Avg Overhead/case ((pred - opt) / opt)    = 469.16%


Adding prediction probabilities...

Saving full predictions to './SVC_RBF_results/tie_tol/50/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 363 wrong predictions with time metrics

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

## Linear Kernel

### Tie Breaking and Data Splitting

#### **Balanced train-test split based on Y**

In [280]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2,
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )

# Label‐encode the string targets into integers 0…4  ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [284]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Instantiate & train SVC with RBF kernel
svc_strat_y_linear = SVC(kernel='linear', C=0.8, gamma='auto', probability=True, random_state=42)

svc_strat_y_linear.fit(X_train_scaled, y_train, sample_weight=weights_train)




SVC(C=0.8, gamma='auto', kernel='linear', probability=True, random_state=42)

Classification report and Feature Importances

In [285]:
# Reversing mapping from integer labels to algorithm names
y_pred_int = svc_strat_y_linear.predict(X_test_scaled)
y_pred_str = le.inverse_transform(y_pred_int)


# Evaluate
y_pred = svc_strat_y_linear.predict(X_test)
print("Classification Report:")
print(classification_report(y_test_str, y_pred_str, sample_weight=weights_test, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

       AltRB       0.10      0.07      0.08 1548.998634966601
    Pivot_Br       0.12      0.05      0.07 3764.789222512713
        S_Br       0.23      0.25      0.24 3152.339561854101
        UMKP       0.39      0.06      0.11 1449.3346678647963
        kPEX       0.32      0.67      0.44 3378.3977986771524

    accuracy                           0.26 13293.859885875361
   macro avg       0.23      0.22      0.19 13293.859885875361
weighted avg       0.22      0.26      0.21 13293.859885875361



#### Inference on 20% dataset

In [286]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=svc_strat_y_linear,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_linear_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_linear_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 49.52%

Precision by Algorithm:
  UMKP: 62.50% (8 predictions)
  AltRB: 27.27% (66 predictions)
  kPEX: 68.61% (137 predictions)
  Pivot_Br: 0.00% (11 predictions)
  S_Br: 45.96% (198 predictions)

Recall by Algorithm:
  UMKP: 36.73% (18/49 cases)
  AltRB: 50.00% (30/60 cases)
  kPEX: 62.79% (135/215 cases)
  Pivot_Br: 43.23% (67/155 cases)
  S_Br: 69.80% (104/149 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 68699.90
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.423
  Avg Overhead/case ((pred - opt) / opt)    = 383.16%


Adding prediction probabilities...

Saving full predictions to './SVC_linear_results/tie_tol/50/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 212 wrong predictions with time metrics

Top 

#### Re-evalute on entire dataset

In [287]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=svc_strat_y_linear,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_linear_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_linear_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 53.03%

Precision by Algorithm:
  UMKP: 55.56% (18 predictions)
  AltRB: 37.00% (100 predictions)
  kPEX: 67.15% (274 predictions)
  Pivot_Br: 20.69% (29 predictions)
  S_Br: 48.97% (339 predictions)

Recall by Algorithm:
  UMKP: 35.00% (35/100 cases)
  AltRB: 48.28% (56/116 cases)
  kPEX: 68.32% (248/363 cases)
  Pivot_Br: 50.00% (113/226 cases)
  S_Br: 73.83% (189/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 117186.00
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.752
  Avg Overhead/case ((pred - opt) / opt)    = 1756.01%


Adding prediction probabilities...

Saving full predictions to './SVC_linear_results/tie_tol/50/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 357 wrong predictions with time me

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

#### **Balanced train-test split based on both Y and k**

In [288]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2,
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )

# Label‐encode the string targets into integers 0…4  ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [300]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Instantiate & train SVC with linear kernel
svc_strat_y_K_linear = SVC(kernel='linear', C=0.5, gamma='auto', probability=True, random_state=42)

svc_strat_y_K_linear.fit(X_train_scaled, y_train, sample_weight=weights_train)



SVC(C=0.5, gamma='auto', kernel='linear', probability=True, random_state=42)

Classification report and Feature Importances

In [301]:

# Reversing mapping from integer labels to algorithm names
y_pred_int = svc_strat_y_K_linear.predict(X_test_scaled)
y_pred_str = le.inverse_transform(y_pred_int)


# Evaluate
y_pred = svc_strat_y_K_linear.predict(X_test)
print("Classification Report:")
print(classification_report(y_test_str, y_pred_str, sample_weight=weights_test, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

       AltRB       0.49      0.00      0.01 1483.3530488672318
    Pivot_Br       0.23      0.21      0.22 3582.073469039411
        S_Br       0.34      0.71      0.46 1849.663881911921
        UMKP       0.29      0.22      0.25 786.182176800659
        kPEX       0.52      0.51      0.52 4679.182576808955

    accuracy                           0.38 12380.455153428178
   macro avg       0.38      0.33      0.29 12380.455153428178
weighted avg       0.39      0.38      0.35 12380.455153428178



#### Inference on 20% dataset

In [302]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=svc_strat_y_K_linear,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_linear_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_linear_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 46.43%

Precision by Algorithm:
  UMKP: 22.22% (45 predictions)
  AltRB: 0.00% (2 predictions)
  kPEX: 75.51% (98 predictions)
  Pivot_Br: 24.00% (50 predictions)
  S_Br: 44.00% (225 predictions)

Recall by Algorithm:
  UMKP: 52.83% (28/53 cases)
  AltRB: 26.09% (18/69 cases)
  kPEX: 53.36% (119/223 cases)
  Pivot_Br: 56.77% (88/155 cases)
  S_Br: 80.82% (118/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 79444.00
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.712
  Avg Overhead/case ((pred - opt) / opt)    = 506.53%


Adding prediction probabilities...

Saving full predictions to './SVC_linear_results/tie_tol/50/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 225 wrong predictions with time metrics

Top

#### Re-evalute on entire dataset

In [303]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=svc_strat_y_K_linear,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_linear_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_linear_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.79%

Precision by Algorithm:
  UMKP: 38.96% (77 predictions)
  AltRB: 50.00% (18 predictions)
  kPEX: 72.93% (181 predictions)
  Pivot_Br: 26.26% (99 predictions)
  S_Br: 49.09% (385 predictions)

Recall by Algorithm:
  UMKP: 52.00% (52/100 cases)
  AltRB: 28.45% (33/116 cases)
  kPEX: 58.95% (214/363 cases)
  Pivot_Br: 55.75% (126/226 cases)
  S_Br: 82.42% (211/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 128960.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.928
  Avg Overhead/case ((pred - opt) / opt)    = 2216.41%


Adding prediction probabilities...

Saving full predictions to './SVC_linear_results/tie_tol/50/expand_and_split/all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 374 wrong predictions with time m

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


#### **Balanced train-test split based on Y**

In [304]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [305]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties

)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [308]:
# Instantiate & train SVC with RBF kernel
svc_holdout_strat_y = SVC(kernel='linear', C=0.75, gamma='auto', probability=True, random_state=42)

svc_holdout_strat_y.fit(X_train_scaled, y_train_, sample_weight=weights_train_)


SVC(C=0.75, gamma='auto', kernel='linear', probability=True, random_state=42)

#### Inference on 20% dataset

In [309]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=svc_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_linear_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_linear_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 49.36%

Precision by Algorithm:
  UMKP: 50.00% (2 predictions)
  AltRB: 31.82% (22 predictions)
  kPEX: 64.41% (59 predictions)
  Pivot_Br: 100.00% (1 predictions)
  S_Br: 41.67% (72 predictions)

Recall by Algorithm:
  UMKP: 30.00% (6/20 cases)
  AltRB: 47.83% (11/23 cases)
  kPEX: 63.01% (46/73 cases)
  Pivot_Br: 42.22% (19/45 cases)
  S_Br: 66.67% (34/51 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 25770.10
  Total Optimal Time   = 16592.90
  Time Ratio (pred/opt)= 1.553
  Avg Overhead/case ((pred - opt) / opt)    = 9481.84%


Adding prediction probabilities...

Saving full predictions to './SVC_linear_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 79 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
           dataset  k_value      w

#### Re-evalute on entire dataset

In [310]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=svc_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_linear_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_linear_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 54.08%

Precision by Algorithm:
  UMKP: 40.00% (10 predictions)
  AltRB: 45.98% (87 predictions)
  kPEX: 64.77% (281 predictions)
  Pivot_Br: 11.11% (9 predictions)
  S_Br: 49.33% (373 predictions)

Recall by Algorithm:
  UMKP: 31.00% (31/100 cases)
  AltRB: 50.86% (59/116 cases)
  kPEX: 68.87% (250/363 cases)
  Pivot_Br: 49.56% (112/226 cases)
  S_Br: 80.86% (207/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 129962.10
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.943
  Avg Overhead/case ((pred - opt) / opt)    = 2261.29%


Adding prediction probabilities...

Saving full predictions to './SVC_linear_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 349 wrong predictions with time metrics

#### **Balanced train-test split based on Y and K**

In [311]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [312]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties

)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [313]:
# Instantiate & train SVC with RBF kernel
svc_holdout_strat_y_K = SVC(kernel='linear', C=1.0, gamma='auto', probability=True, random_state=42)

svc_holdout_strat_y_K.fit(X_train_scaled, y_train_, sample_weight=weights_train_)


SVC(gamma='auto', kernel='linear', probability=True, random_state=42)

#### Inference on 20% dataset

In [314]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=svc_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_linear_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_linear_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 56.08%

Precision by Algorithm:
  UMKP: 66.67% (3 predictions)
  AltRB: 42.86% (7 predictions)
  kPEX: 62.69% (67 predictions)
  Pivot_Br: 100.00% (1 predictions)
  S_Br: 50.00% (70 predictions)

Recall by Algorithm:
  UMKP: 45.00% (9/20 cases)
  AltRB: 39.13% (9/23 cases)
  kPEX: 73.97% (54/73 cases)
  Pivot_Br: 53.33% (24/45 cases)
  S_Br: 80.39% (41/51 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 25258.50
  Total Optimal Time   = 11002.60
  Time Ratio (pred/opt)= 2.296
  Avg Overhead/case ((pred - opt) / opt)    = 10438.09%


Adding prediction probabilities...

Saving full predictions to './SVC_linear_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 65 wrong predictions with time metrics

Top 5 heavies

#### Re-evalute on entire dataset

In [315]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=svc_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./SVC_linear_results/tie_tol/50/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./SVC_linear_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 53.82%

Precision by Algorithm:
  UMKP: 37.04% (27 predictions)
  AltRB: 42.65% (68 predictions)
  kPEX: 64.47% (304 predictions)
  Pivot_Br: 16.67% (6 predictions)
  S_Br: 48.73% (355 predictions)

Recall by Algorithm:
  UMKP: 36.00% (36/100 cases)
  AltRB: 40.52% (47/116 cases)
  kPEX: 70.80% (257/363 cases)
  Pivot_Br: 50.44% (114/226 cases)
  S_Br: 79.30% (203/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 123092.50
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.840
  Avg Overhead/case ((pred - opt) / opt)    = 2150.51%


Adding prediction probabilities...

Saving full predictions to './SVC_linear_results/tie_tol/50/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 351 wrong predictions with time metri

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br